<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/10-Adding_Reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improving Retrieval with Cohere Reranking

This notebook demonstrates how to add **Cohere reranking** as a post-processing step on top of LlamaIndex vector retrieval to improve the quality of retrieved context for RAG pipelines.

## What You'll Learn
- How to load a pre-built vector store from HuggingFace Hub
- How to configure `CohereRerank` as a node post-processor in LlamaIndex
- How to run a query with reranking and inspect the reranked results
- How to evaluate retrieval quality (hit rate and MRR) with and without reranking using `RetrieverEvaluator`

## Prerequisites
- Familiarity with RAG pipelines and vector stores (see notebooks 02–09)
- A working OpenAI API key (for embeddings and LLM)
- A working Cohere API key (for the reranker)
- Basic understanding of LlamaIndex query engines and retrievers

## Install Packages and Setup Variables

In [ ]:
!pip install -qU llama-index==0.14.16 openai==2.30.0 google-genai==1.70.0 llama-index-embeddings-openai==0.5.2 \
                chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 opentelemetry-api==1.38.0\
                llama-index-llms-openai==0.6.26 llama-index-finetuning==0.4.2 llama-index-embeddings-huggingface==0.7.0 \
                cohere==5.21.1 llama-index-postprocessor-cohere-rerank==0.5.1 llama-index-embeddings-cohere==0.8.0 \
                llama-index-llms-google-genai==0.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import asyncio
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os

# Set the following API keys in the Python environment. They will be used later.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# os.environ["CO_API_KEY"] = "YOUR_COHERE_API_KEY"
# os.environ["COHERE_API_KEY"] = "YOUR_COHERE_API_KEY"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    os.environ["CO_API_KEY"] = userdata.get('COHERE_API_KEY')
    os.environ["COHERE_API_KEY"] = userdata.get('COHERE_API_KEY')
    cohere_key = userdata.get('COHERE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")
    os.environ["CO_API_KEY"] = os.environ.get("COHERE_API_KEY", "")
    cohere_key = os.environ.get("COHERE_API_KEY", "")

# Initialize Embedding and LLM Models

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

### Download vector store

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os
vs_zip = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir="./")
with zipfile.ZipFile(vs_zip, 'r') as zf:
    zf.extractall(".")

vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create the vector index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_vector_store(vector_store)

# Query Dataset

In [ ]:
from llama_index.postprocessor.cohere_rerank import CohereRerank

# Define the Cohere Reranking object to return only the two highest-ranking chunks.
cohere_rerank3 = CohereRerank(top_n=2, api_key=cohere_key, model='rerank-english-v3.0')

In [ ]:
# Define the query engine with the LLM for generating the final answer
# and the embedding model for retrieving related nodes.
# The `node_postprocessors` step will be applied to the retrieved nodes.
query_engine = index.as_query_engine(
    similarity_top_k=5, node_postprocessors=[cohere_rerank3]
)

res = query_engine.query("Write about Retrieval Augmented Generation?")

In [ ]:
print(res.response)

Retrieval-Augmented Generation (RAG) is a hybrid approach that combines information retrieval with large language model (LLM) generation to produce more accurate, up-to-date, and context-grounded responses. It addresses common LLM problems such as outdated knowledge and factual fabrication by supplying relevant external content at generation time rather than relying solely on the model’s parametric memory.

Core components
- Retrieval component
  - Purpose: extract relevant content from external knowledge sources (documents, databases, corpora) given a user query.
  - Phases:
    - Indexing: organize documents for efficient lookup. This can use sparse methods (inverted indexes) or dense methods (vector encodings of document chunks).
    - Searching: query the index to fetch candidate documents; optional rerankers refine ordering of retrieved items to improve relevance.
  - Design choices: how to chunk documents, what embedding types to use for semantic representation, and whether to ap

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 1c324686-fad7-4a41-bfd3-d44f9612ca91
Title	 Evaluation of Retrieval-Augmented Generation: A Survey:Research Paper Information
Text	 Authors: Hao Yu, Aoran Gan, Kai Zhang, Shiwei Tong, Qi Liu, and Zhaofeng Liu.Numerous studies of Retrieval-Augmented Generation (RAG) systems have emerged from various perspectives since the advent of Large Language Models (LLMs). The RAG system comprises two primary components: Retrieval and Generation. The retrieval component aims to extract relevant information from various external knowledge sources. It involves two main phases, indexing and searching. Indexing organizes documents to facilitate efficient retrieval, using either inverted indexes for sparse retrieval or dense vector encoding for dense retrieval. The searching component utilizes these indexes to fetch relevant documents based on the user's query, often incorporating the optional rerankers to refine the ranking of the retrieved documents. The generation component utilizes the retr

## Optional: Reranking Score Comparison

In [ ]:
# Compare relevance scores before and after Cohere reranking
print("=== Node Scores After Cohere Reranking ===\n")
print(f"{'Rank':<6} {'Score':>8}  {'Text preview'}")
print("-" * 70)
for rank, node in enumerate(res.source_nodes, 1):
    score = f"{node.score:.4f}" if node.score is not None else "  N/A "
    preview = node.text[:80].replace("\n", " ")
    print(f"{rank:<6} {score:>8}  {preview}...")

print(f"\nTotal nodes returned after reranking: {len(res.source_nodes)}")
print("Note: Cohere reranker re-scores all retrieved nodes and returns the top-N most relevant.")

=== Node Scores After Cohere Reranking ===

Rank      Score  Text preview
----------------------------------------------------------------------
1        0.9970  Authors: Hao Yu, Aoran Gan, Kai Zhang, Shiwei Tong, Qi Liu, and Zhaofeng Liu.Num...
2        0.9967  Generative large language models are prone to producing outdated information or ...

Total nodes returned after reranking: 2
Note: Cohere reranker re-scores all retrieved nodes and returns the top-N most relevant.


# Evaluate Cohere rerank

In [ ]:
from huggingface_hub import hf_hub_download
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

# Download the evaluation dataset
hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="rag_eval_dataset_question_context_subset_50.json", repo_type="dataset", local_dir="./")
rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset_question_context_subset_50.json")

(…)_dataset_question_context_subset_50.json: 0.00B [00:00, ?B/s]

In [ ]:
import pandas as pd
from llama_index.core.evaluation import RetrieverEvaluator

def display_results_retriever(name, eval_results):
    """Display retriever evaluation results as a formatted string."""
    metric_dicts = [r.metric_vals_dict for r in eval_results]
    full_df = pd.DataFrame(metric_dicts)
    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()
    return f"{name} | Hit Rate: {hit_rate:.4f} | MRR: {mrr:.4f}"

# We can evaluate the retrievers with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(
        similarity_top_k=i, node_postprocessors=[cohere_rerank3]
    )
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

Retriever top_2 | Hit Rate: 0.6000 | MRR: 0.5700
Retriever top_4 | Hit Rate: 0.6800 | MRR: 0.5933
Retriever top_6 | Hit Rate: 0.7000 | MRR: 0.5973
Retriever top_8 | Hit Rate: 0.7800 | MRR: 0.6084
Retriever top_10 | Hit Rate: 0.8000 | MRR: 0.6106


## Optional: With vs Without Reranking

In [ ]:
# Compare the response quality with and without the Cohere reranker
llm = Settings.llm
test_query = "What are the key differences between LLaMA 1 and LLaMA 2?"

print(f"Query: {test_query}\n")
print("=" * 65)

# Without reranking
try:
    plain_engine = index.as_query_engine(similarity_top_k=5, llm=llm)
    plain_res = plain_engine.query(test_query)
    print("WITHOUT reranking:")
    print(str(plain_res)[:500])
    print(f"(Retrieved {len(plain_res.source_nodes)} nodes)")
except Exception as e:
    print(f"Without reranking error: {e}")

print("\n" + "-" * 65)

# With reranking
try:
    rerank_engine = index.as_query_engine(
        similarity_top_k=10,
        node_postprocessors=[CohereRerank(api_key=os.environ.get("COHERE_API_KEY",""), top_n=3)],
        llm=llm,
    )
    rerank_res = rerank_engine.query(test_query)
    print("WITH Cohere reranking (top_n=3 from 10):")
    print(str(rerank_res)[:500])
    print(f"(Retrieved {len(rerank_res.source_nodes)} nodes after reranking)")
except Exception as e:
    print(f"With reranking error: {e}")

Query: What are the key differences between LLaMA 1 and LLaMA 2?

WITHOUT reranking:
- Architecture: LLaMA 2 includes architectural tweaks not present in the original LLaMA (one named change is Grouped Query Attention).
- Pretraining scale: LLaMA 2 was pre-trained on a much larger token corpus (noted as 2 trillion tokens) compared with the original LLaMA.
- Implementation lineage: the original LLaMA implementation and subsequent LLaMA 2 work were integrated into toolkits using codebases such as GPT‑NeoX and adapted into various implementations (including Flax), reflecting evolut
(Retrieved 5 nodes)

-----------------------------------------------------------------
WITH Cohere reranking (top_n=3 from 10):
- Grouped Query Attention (GQA): LLaMA 2 introduces architectural tweaks such as Grouped Query Attention that modify the attention mechanism versus the original LLaMA.

- Much larger pre‑training corpus: LLaMA 2 was pre‑trained on a substantially larger token budget (reported as ~2 tri